In [0]:
%sql
---This creates the comparison Table
CREATE OR REPLACE TABLE workspace.analytics.stedi_model_comparison AS

WITH base AS (
  SELECT
    SUM(CASE WHEN actual_label = predicted_label THEN 1 ELSE 0 END) AS correct,
    COUNT(*) AS total,
    SUM(CASE WHEN actual_label = 1 AND predicted_label = 1 THEN 1 ELSE 0 END) AS tp,
    SUM(CASE WHEN predicted_label = 1 THEN 1 ELSE 0 END) AS predicted_positive,
    SUM(CASE WHEN actual_label = 1 THEN 1 ELSE 0 END) AS actual_positive
  FROM workspace.analytics.stedi_predictions_capstone
)

SELECT
  'RandomForest_v1' AS model_name,
  correct / total AS accuracy,
  tp / predicted_positive AS precision,
  tp / actual_positive AS recall,
  2 * (tp / predicted_positive) * (tp / actual_positive)
    / ((tp / predicted_positive) + (tp / actual_positive)) AS f1_score
FROM base;

In [0]:
%sql
SHOW TABLES IN workspace.analytics;

In [0]:
%sql
SELECT COUNT(*) 
FROM workspace.analytics.stedi_predictions_capstone;

In [0]:
%sql
SELECT *
FROM workspace.analytics.stedi_model_comparison;

In [0]:
%sql
CREATE TABLE workspace.analytics.stedi_roc_curve_v1 AS

WITH counts AS (
  SELECT
    SUM(CASE WHEN actual_label = 1 AND predicted_label = 1 THEN 1 ELSE 0 END) AS tp,
    SUM(CASE WHEN actual_label = 0 AND predicted_label = 0 THEN 1 ELSE 0 END) AS tn,
    SUM(CASE WHEN actual_label = 0 AND predicted_label = 1 THEN 1 ELSE 0 END) AS fp,
    SUM(CASE WHEN actual_label = 1 AND predicted_label = 0 THEN 1 ELSE 0 END) AS fn
  FROM workspace.analytics.stedi_predictions_capstone
)

SELECT
  'RandomForest_v1' AS model_name,
  0.0 AS false_positive_rate,
  0.0 AS true_positive_rate

UNION ALL

SELECT
  'RandomForest_v1',
  CAST(fp AS DOUBLE) / (fp + tn),
  CAST(tp AS DOUBLE) / (tp + fn)
FROM counts

UNION ALL

SELECT
  'RandomForest_v1',
  1.0,
  1.0;

In [0]:
%sql
SELECT *
FROM workspace.analytics.stedi_roc_curve_v1;